In [2]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

def remove_params(s):
    pattern = r'\d+(\.\d+)?[BbMm](\d)?'
    cleaned_str = re.sub(pattern, '', s)
    if cleaned_str[-1]=='-':
        cleaned_str = cleaned_str[:-1]
    cleaned_str = cleaned_str.replace("--","-")
    return cleaned_str

def standard_name(s, slash=True):
    if str(s)=='nan':
        return ''
    else:
        if not slash:
            return s.lower().replace("-hf","")
        else:
            if "/" in s:
                return s.split("/")[1].lower().replace("-hf","")
            else:
                return s.lower().replace("-hf","")

def are_strings_equivalent(str1, str2):
    cleaned_str1 = remove_params(str1)
    cleaned_str2 = remove_params(str2)
    bool1 = cleaned_str1 == cleaned_str2
    return bool1 
    
def get_families(data, min=2):
    if type(data)==list:
        models = np.unique(data).tolist()
    else:
        models = np.unique(list(data.model)).tolist()
    D = (np.array([[are_strings_equivalent(m1, m2) for m1 in models] for m2 in tqdm(models)]))
    
    families = []
    while len(models)>0:
        indices = [j for j,bool in enumerate(D[0]) if bool]
        D = np.delete(D, indices, axis=0)
        D = np.delete(D, indices, axis=1)
        families.append(np.array(models)[indices].tolist())
    
        for m in np.array(models)[indices].tolist():
            models.remove(m)
    
    families = [f for f in families if len(f)>=min]
    #families_instruct = [f for f in families if 'chat' in f[0].lower() or 'instruct' in f[0].lower() or '-it' in f[0][-4:].lower()]
    #families_base = [f for f in families if f not in families_instruct]

    families = [np.sort(f).tolist() for f in families]
    #families_instruct = [np.sort(f).tolist() for f in families if data.loc[data.Model==f[0]]['T'].iloc[0]=='💬']
    #families_base = [np.sort(f).tolist() for f in families if data.loc[data.Model==f[0]]['T'].iloc[0]=='🟢']
    return families#, families_base, families_instruct

def replace_underscores(input_string):
    """
    Replaces "__" with "_" and "_" with "" only if "_" is at the end of the string.

    Args:
        input_string (str): The input string to process.

    Returns:
        str: The processed string with replacements.
    """
    if input_string.endswith("_"):
        input_string = input_string[:-1]
    else:
        # Replace "__" with "_" only
        input_string = input_string.replace("__", "_")
    
    return input_string
    
def get_family_name(strings):
    # Start with the shortest string in the list
    shortest_string = min(strings, key=len)
    length = len(shortest_string)
    
    # Iterate over all possible substrings of the shortest string
    for sub_len in range(length, 0, -1):  # Start with the longest substrings
        for i in range(length - sub_len + 1):
            substring = shortest_string[i:i + sub_len]
            # Check if this substring is in all other strings
            if all(substring in string for string in strings):
                return substring
    
    return ""  # Return an empty string if no common substring is found

In [2]:
new_df = pd.read_parquet("hf://datasets/open-llm-leaderboard/contents/data/train-00000-of-00001.parquet")
new_df = new_df.loc[:,['fullname','Base Model','IFEval','BBH','MATH Lvl 5','MMLU-PRO','GPQA','MUSR','Submission Date']]
new_df = new_df.dropna()
new_df.loc[:,'Submission Date'] = pd.to_datetime(new_df.loc[:,'Submission Date'])
new_df.iloc[:,2:-1] = new_df.iloc[:,2:-1]/100
new_df.columns = ['model', 'base_model', 'ifeval', 'bbh', 'math', 'mmlu-pro', 'gpqa', 'musr', 'date']
new_df = new_df.sort_values(by='date', ascending=False).drop_duplicates(subset='model', keep='first')
new_df = new_df.sort_values(by='model').reset_index(drop=True).iloc[:,:-1]
new_df.to_csv("new_processed.csv")
old_df = pd.read_parquet("hf://datasets/open-llm-leaderboard-old/contents/data/train-00000-of-00001-96886cb34a7bc800.parquet")
old_df = old_df.loc[:,['fullname','ARC','HellaSwag','MMLU','TruthfulQA','Winogrande', 'GSM8K','date']]
old_df = old_df.dropna()
old_df.loc[:,'date'] = pd.to_datetime([s.split("T")[0] for s in old_df.date])
old_df.columns = ['model', 'arc', 'hellaswag', 'mmlu', 'truthfulqa', 'winogrande', 'gsm8k', 'date']
old_df.iloc[:,1:-1] = old_df.iloc[:,1:-1]/100
old_df = old_df.sort_values(by='date', ascending=False).drop_duplicates(subset='model', keep='first')
old_df = old_df.sort_values(by='model').reset_index(drop=True).iloc[:,:-1]
old_df.to_csv("old_processed.csv")

In [3]:
df = old_df.merge(new_df, on='model', how='outer')

families = get_families(df, min=1)

family_names = [get_family_name([replace_underscores(remove_params(f)) for f in ff]) for ff in families]

families_dict = {}
for i,fam in enumerate(families):
    for m in fam:
        families_dict[m] = family_names[i]

df = df.loc[[m in families_dict.keys() for m in df.model]]

df.loc[:,'family'] = [families_dict[m] for m in df['model']]
df = df.loc[:,['model', 'base_model', 'family', 'arc', 'hellaswag', 'mmlu', 'truthfulqa', 'winogrande','gsm8k', 'ifeval', 'bbh', 'math', 'mmlu-pro', 'gpqa','musr']]
df['model'] = [standard_name(m) for m in df.model]
df['base_model'] = [standard_name(m) for m in df.base_model]
df['family'] = [standard_name(m) for m in df.family]
#df = df.loc[[m!='' for m in df.base_model]]
#df = df.reset_index(drop=True)
df.to_csv("df.csv")

  0%|          | 0/9446 [00:00<?, ?it/s]

In [4]:
#bms = np.unique(df.base_model)
#bms[['llama-2' in m for m in bms]]

In [5]:
df_sloth = pd.read_csv("data_v2.csv")
df_sloth = df_sloth.loc[:,['Model','Family2','#Params (B)','Pretraining Data Size (T)','FLOPs (1E21)',
                           'ARC','HellaSwag','MMLU','TruthfulQA','Winogrande','GSM8K','IFEval','BBH','MATH Lvl 5','MMLU-PRO','GPQA','MUSR']]
df_sloth.columns = ['model','family','size','tokens','flops','arc', 'hellaswag', 'mmlu', 'truthfulqa', 'winogrande','gsm8k', 'ifeval', 'bbh', 'math', 'mmlu-pro', 'gpqa','musr']

In [8]:
df = df.loc[[mb in list(df_sloth.model) and m not in list(df_sloth.model) for m,mb in zip(df.model,df.base_model)]]
print(df.shape)
df['size'] = None
df['tokens'] = None
df['flops'] = None

vec = []
for i,m in tqdm(enumerate(df.base_model)):
    vec.append(np.array(df_sloth.loc[df_sloth.model==m,['size','tokens','flops']]).squeeze())
vec = np.array(vec)
df.iloc[:,-3:] = vec

(116, 15)


0it [00:00, ?it/s]

In [9]:
df_full = pd.concat((df_sloth, df), axis=0).drop(['base_model'], axis=1)
df_full_v1 = df_full.loc[:,['model', 'family', 'size', 'tokens', 'flops',
                            'arc', 'hellaswag','mmlu', 'truthfulqa', 'winogrande', 'gsm8k']].dropna()
df_full_v2 = df_full.loc[:,['model', 'family', 'size', 'tokens', 'flops',
                            'ifeval', 'bbh', 'math','mmlu-pro', 'gpqa', 'musr']].dropna()
df_full = df_full.dropna()

In [10]:
df_full.shape,df_full_v1.shape,df_full_v2.shape

((98, 17), (168, 11), (216, 11))

In [11]:
df_full.to_csv("df_full.csv")
df_full_v1.to_csv("df_full_v1.csv")
df_full_v2.to_csv("df_full_v2.csv")

In [4]:
df_full = pd.read_csv("df_full.csv").iloc[:,[1,2]]
df_full_v1 = pd.read_csv("df_full_v1.csv").iloc[:,[1,2]]
df_full_v2 = pd.read_csv("df_full_v2.csv").iloc[:,[1,2]]



In [11]:
df_full.shape[0], np.unique(df_full.family).shape[0]

(98, 50)

In [12]:
df_full_v1.shape[0], np.unique(df_full_v1.family).shape[0]

(168, 75)

In [13]:
df_full_v2.shape[0], np.unique(df_full_v2.family).shape[0]

(216, 146)

In [3]:
df_full_v1 = pd.read_csv("df_full_v1.csv").iloc[:,[1,2]]
df_full_v2 = pd.read_csv("df_full_v2.csv").iloc[:,[1,2]]
df_full_v1['leaderboard-1']=True
df_full_v2['leaderboard-2']=True

In [32]:
data = pd.merge(df_full_v1,df_full_v2,on=['model','family'],how='outer').fillna(False)

/tmp/ipykernel_334962/1282567173.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = pd.merge(df_full_v1,df_full_v2,on=['model','family'],how='outer').fillna(False)


In [35]:
data.model = [m.replace('_','-')[:30] for m in data.model]
data.family = [m.replace('_','-')[:30] for m in data.family]

In [36]:
print(data.to_latex())

\begin{tabular}{lllrr}
\toprule
 & model & family & leaderboard-1 & leaderboard-2 \\
\midrule
0 & athena-gemma-2-2b-it & athena-gemma-2-it & False & True \\
1 & bio-medical-llama-3-8b & bio-medical-llama-3 & False & True \\
2 & bloom & bloom & True & False \\
3 & bloom-1b1 & bloom & True & True \\
4 & bloom-3b & bloom & True & True \\
5 & bloom-560m & bloom & True & True \\
6 & bloom-7b1 & bloom & True & True \\
7 & blossom-v5.1-34b & blossom-v5.1 & True & True \\
8 & blossom-v5.1-9b & blossom-v5.1 & False & True \\
9 & braincog-8b-0.1-instruct & braincog-0.1-instruct & False & True \\
10 & calme-2.1-qwen2-72b & calme-2.1-qwen2 & False & True \\
11 & calme-2.1-qwen2-7b & calme-2.1-qwen2 & False & True \\
12 & calme-2.2-llama3-70b & calme-2.2-llama3 & False & True \\
13 & calme-2.2-qwen2-72b & calme-2.2-qwen2 & False & True \\
14 & calme-2.2-qwen2-7b & calme-2.2-qwen2 & False & True \\
15 & calme-2.3-llama3-70b & calme-2.3-llama3 & False & True \\
16 & calme-2.3-qwen2-72b & calme-2.3-qw